# Pyro BNN Surrogate + BoTorch ile Bayesçi Optimizasyon

Bu notebook, pahalı bir üretim/proses fonksiyonunu az sayıda deneyle optimize etmek için:

`deneyler → Pyro BNN surrogate → posterior samples → BoTorch qLogEI → yeni deney`

akışını gösterir.

Örnek iki proses parametresini optimize eder. Gerçek uygulamada bunlar sıcaklık/basınç, hız/besleme, kaynak parametreleri, enerji ayarları veya simülasyon girdileri olabilir.

> Not: BoTorch'un Monte Carlo acquisition fonksiyonları GP zorunluluğu getirmez. `Model.posterior()` tarafından örneklenebilir (`rsample`) bir posterior sağlanması yeterlidir. Burada Pyro posterior predictive örneklerini `EnsembleModel` üzerinden BoTorch'a bağlıyoruz.


In [ ]:
# Gerekirse:
# %pip install torch pyro-ppl botorch numpy pandas matplotlib

from typing import Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import Tensor
import torch.nn as nn

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoGuide, AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample

from botorch.models.ensemble import EnsembleModel
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim.optimize import optimize_acqf

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)

# BoTorch için double precision önerilir.
torch.set_default_dtype(torch.float64)


## 1. Pahalı black-box proses

Fonksiyonun formülünü algoritmanın bildiğini varsaymıyoruz. Burada yalnızca sentetik deney üretmek için kullanıyoruz.

Amaç **maksimizasyon**: yüksek kalite/performans, düşük enerji cezası.


In [ ]:
def true_process(X: Tensor) -> Tensor:
    x1 = X[..., 0]
    x2 = X[..., 1]

    main_peak = 1.35 * torch.exp(
        -((x1 - 0.68)**2 / 0.025 + (x2 - 0.34)**2 / 0.035)
    )
    secondary_peak = 0.45 * torch.exp(
        -((x1 - 0.28)**2 / 0.06 + (x2 - 0.72)**2 / 0.08)
    )
    interaction = 0.08 * torch.sin(9*x1) * torch.cos(7*x2)
    energy_penalty = 0.22*x1 + 0.12*x2

    return main_peak + secondary_peak + interaction - energy_penalty

def observe(X: Tensor, noise_sd: float = 0.025) -> Tensor:
    y = true_process(X) + noise_sd * torch.randn(X.shape[:-1])
    return y.unsqueeze(-1)

bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

train_X = torch.rand(16, 2)
train_Y = observe(train_X)

print("Başlangıç en iyi gözlem:", float(train_Y.max()))


In [ ]:
grid_n = 70
g1 = torch.linspace(0, 1, grid_n)
g2 = torch.linspace(0, 1, grid_n)
G1, G2 = torch.meshgrid(g1, g2, indexing="ij")
grid = torch.stack([G1.reshape(-1), G2.reshape(-1)], dim=-1)
Z = true_process(grid).reshape(grid_n, grid_n)

plt.figure(figsize=(7, 5))
plt.contourf(G1.numpy(), G2.numpy(), Z.numpy(), levels=25)
plt.scatter(train_X[:, 0].numpy(), train_X[:, 1].numpy(), marker="x")
plt.xlabel("Proses parametresi 1")
plt.ylabel("Proses parametresi 2")
plt.title("Gerçek fonksiyon (algoritma bunu bilmiyor)")
plt.show()


## 2. Pyro ile Bayesçi sinir ağı surrogate

Ağın iki katmanındaki ağırlık ve bias'lara önsel dağılım koyuyoruz. `AutoDiagonalNormal`, mean-field variational posterior üretir.

BoTorch açısından kritik nokta posteriorun biçimi değil, yeni karar noktalarında **posterior predictive sample** üretebilmesidir.


In [ ]:
class BayesianProcessNN(PyroModule):
    Y_SITE = "y"

    def __init__(self, in_features=2, hidden=20):
        super().__init__()

        self.h = PyroModule[nn.Linear](in_features, hidden)
        self.h.weight = PyroSample(
            dist.Normal(0.0, 1.0).expand([hidden, in_features]).to_event(2)
        )
        self.h.bias = PyroSample(
            dist.Normal(0.0, 1.0).expand([hidden]).to_event(1)
        )

        self.out = PyroModule[nn.Linear](hidden, 1)
        self.out.weight = PyroSample(
            dist.Normal(0.0, 1.0).expand([1, hidden]).to_event(2)
        )
        self.out.bias = PyroSample(
            dist.Normal(0.0, 1.0).expand([1]).to_event(1)
        )

    def forward(self, x: Tensor, y: Optional[Tensor] = None) -> Tensor:
        # Acquisition optimizasyonunda X'e göre gradient gereklidir.
        torch.set_grad_enabled(True)

        mean = self.out(torch.tanh(self.h(x))).squeeze(-1)
        sigma = pyro.sample("sigma", dist.LogNormal(-3.0, 0.45))

        with pyro.plate("data", x.shape[0]):
            pyro.sample(
                self.Y_SITE,
                dist.StudentT(df=4.0, loc=mean, scale=sigma),
                obs=y,
            )
        return mean


In [ ]:
def fit_svi(
    model: PyroModule,
    guide: AutoGuide,
    X: Tensor,
    Y: Tensor,
    epochs: int = 700,
) -> list[float]:
    svi = SVI(
        model,
        guide,
        pyro.optim.Adam({"lr": 0.02}),
        loss=Trace_ELBO(),
    )
    losses = []
    for epoch in range(epochs):
        losses.append(svi.step(X, Y.squeeze(-1)) / len(X))
    return losses


## 3. Pyro posteriorunu BoTorch `EnsembleModel` arayüzüne bağla

`EnsemblePosterior` örnekleri kabaca şu şekildedir:

`batch × posterior_sample × q × output`

Bu wrapper, Pyro'nun `Predictive` çıktısını BoTorch'un acquisition fonksiyonlarının tüketebileceği biçime çevirir.


In [ ]:
class PyroBNNBoTorchModel(EnsembleModel):
    _num_outputs: int

    def __init__(
        self,
        train_X: Tensor,
        train_Y: Tensor,
        num_samples: int = 128,
        hidden: int = 20,
    ):
        super().__init__()
        self._num_outputs = train_Y.shape[-1]
        self.model = BayesianProcessNN(
            in_features=train_X.shape[-1],
            hidden=hidden,
        )
        self.guide = AutoDiagonalNormal(self.model)
        self.num_samples = num_samples

    def forward(self, X: Tensor) -> Tensor:
        # Bu öğretim örneği q=1 aday optimizasyonuna odaklanır.
        if X.shape[-2] != 1:
            raise ValueError("Bu örnek q=1 için tasarlanmıştır.")

        X_eval = X.squeeze(-2)  # batch x d

        predictive = Predictive(
            self.model,
            guide=self.guide,
            num_samples=self.num_samples,
            return_sites=(self.model.Y_SITE,),
        )

        raw = predictive(X_eval)[self.model.Y_SITE]  # s x batch

        # batch x s x q x m
        return raw.transpose(0, 1).unsqueeze(-1).unsqueeze(-1)


In [ ]:
pyro.clear_param_store()

bnn_model = PyroBNNBoTorchModel(train_X, train_Y, num_samples=128)
losses = fit_svi(
    bnn_model.model,
    bnn_model.guide,
    train_X,
    train_Y,
    epochs=900,
)

plt.plot(losses)
plt.xlabel("SVI adımı")
plt.ylabel("ELBO / gözlem")
plt.show()


## 4. BNN posterior ortalaması ve belirsizliği

BoTorch `posterior()` çağrısı artık custom BNN modelimiz üzerinde çalışır.


In [ ]:
with torch.no_grad():
    posterior = bnn_model.posterior(grid.unsqueeze(-2))
    post_mean = posterior.mean.squeeze(-1).squeeze(-1)
    post_var = posterior.variance.squeeze(-1).squeeze(-1)

mean_surface = post_mean.reshape(grid_n, grid_n)
sd_surface = post_var.sqrt().reshape(grid_n, grid_n)

plt.figure(figsize=(7, 5))
plt.contourf(G1.numpy(), G2.numpy(), mean_surface.numpy(), levels=25)
plt.scatter(train_X[:, 0].numpy(), train_X[:, 1].numpy(), marker="x")
plt.xlabel("Proses parametresi 1")
plt.ylabel("Proses parametresi 2")
plt.title("BNN posterior ortalaması")
plt.show()

plt.figure(figsize=(7, 5))
plt.contourf(G1.numpy(), G2.numpy(), sd_surface.numpy(), levels=25)
plt.scatter(train_X[:, 0].numpy(), train_X[:, 1].numpy(), marker="x")
plt.xlabel("Proses parametresi 1")
plt.ylabel("Proses parametresi 2")
plt.title("BNN posterior standart sapması")
plt.show()


## 5. qLogExpectedImprovement ile yeni deney seç

Acquisition function şu dengeyi kurar:

- yüksek beklenen performans → exploitation,
- yüksek epistemik belirsizlik → exploration.

BoTorch'un güncel önerileri doğrultusunda klasik `qExpectedImprovement` yerine sayısal olarak daha kararlı `qLogExpectedImprovement` kullanıyoruz.


In [ ]:
best_f = train_Y.max()

acq = qLogExpectedImprovement(
    model=bnn_model,
    best_f=best_f,
)

candidate, acq_value = optimize_acqf(
    acq_function=acq,
    bounds=bounds,
    q=1,
    num_restarts=10,
    raw_samples=128,
)

new_y = observe(candidate)

print("Önerilen yeni proses ayarı:", candidate.detach().numpy().round(4))
print("Yeni gözlem:", float(new_y))
print("Önceki en iyi:", float(best_f))


## 6. Küçük bir Bayesçi optimizasyon döngüsü

Aşağıdaki hücre modeli her iterasyonda yeniden eğitir. Öğretim amacıyla basit tutulmuştur; büyük projelerde warm-start, daha gelişmiş posteriorlar, batch BO ve Ax orchestration düşünülebilir.


In [ ]:
X_bo = train_X.clone()
Y_bo = train_Y.clone()
history = [float(Y_bo.max())]

for iteration in range(5):
    pyro.clear_param_store()

    surrogate = PyroBNNBoTorchModel(
        X_bo,
        Y_bo,
        num_samples=96,
        hidden=20,
    )
    _ = fit_svi(
        surrogate.model,
        surrogate.guide,
        X_bo,
        Y_bo,
        epochs=550,
    )

    acq = qLogExpectedImprovement(
        model=surrogate,
        best_f=Y_bo.max(),
    )

    candidate, _ = optimize_acqf(
        acq_function=acq,
        bounds=bounds,
        q=1,
        num_restarts=8,
        raw_samples=96,
    )

    Y_new = observe(candidate)
    X_bo = torch.cat([X_bo, candidate.detach()], dim=0)
    Y_bo = torch.cat([Y_bo, Y_new.detach()], dim=0)

    history.append(float(Y_bo.max()))
    print(
        f"Iterasyon {iteration+1}: "
        f"x={candidate.detach().numpy().round(3)}, "
        f"y={float(Y_new):.4f}, "
        f"best={float(Y_bo.max()):.4f}"
    )

plt.plot(history, marker="o")
plt.xlabel("BO iterasyonu")
plt.ylabel("Şimdiye kadarki en iyi gözlem")
plt.title("BNN + BoTorch optimizasyon ilerlemesi")
plt.show()


## 7. Endüstri mühendisliği açısından kullanım alanları

Aynı mimari şu problemlere taşınabilir:

| Pahalı değerlendirme | Karar değişkenleri |
|---|---|
| Ayrık olay simülasyonu | kapasite / personel / buffer |
| Dijital ikiz | proses set-point'leri |
| FEA / CFD | mühendislik tasarım parametreleri |
| Gerçek fabrika deneyi | sıcaklık / basınç / hız |
| Enerji sistemi simülasyonu | kontrol ve işletme ayarları |
| Kalite deneyi | proses reçetesi |

### Ne zaman GP yerine BNN?

Az veri ve düşük boyutta **GP genellikle ilk baseline olmalıdır**. BNN özellikle veri miktarı, boyut veya response karmaşıklığı arttığında anlam kazanır.

Araştırma tasarımında en az şu karşılaştırmayı yapın:

`GP vs BNN vs deterministic NN/deep ensemble`

ve yalnız RMSE değil:
- calibration,
- regret,
- optimuma ulaşmak için gereken deney sayısı,
- final decision quality
ölçün.


## 8. Teknik not

BoTorch'un MC acquisition fonksiyonları custom surrogate'ın GP olmasını gerektirmez. Temel gereksinim `posterior()` metodundan örneklenebilir bir posterior elde edilmesidir. Gradient tabanlı acquisition optimizasyonu için posterior örneklerinden giriş `X`'e geri türev alınabilmesi de gerekir.

Bu notebookta Pyro'nun posterior predictive örnekleri `EnsembleModel` ile bu sözleşmeye uydurulmuştur.
